# FMD lens defect classifier: six architectures

This notebook holds the six architectures trained on the 22-class dataset, one cell each, in the same
form as the four architecture cells you supplied: every cell defines its block and then builds the
model, so the cells can be stepped through and the layers inspected directly.

Four of the six are your architectures, unchanged. Two baselines were added so the comparison covers the
whole matrix.

| # | Architecture | Where it comes from | Accuracy on the 22-class set |
|---|---|---|---|
| 1 | ResNet50 MultiLevel MultiScale | your Architecture 1, unchanged | 88.05 |
| 2 | ResNet50 MultiLevel Attention (SE) | your Architecture 2, unchanged | 87.37 |
| 3 | Inception with attention inside every branch | your Architecture 3, unchanged | 86.01 |
| 4 | Inception with a refined decode path | your Architecture 4, unchanged | 88.05 |
| 5 | ResNet18 | baseline we added | 82.94 |
| 6 | ResNet50 plain | baseline we added | 88.40 |

Training settings are identical for all six: 50 epochs, batch 8, seed 42, input 224x224, SGD with learning
rate 1e-4, momentum 0.9 and Nesterov, on the same train and test split. Per-class recall for all 22
classes, plus training time, is in the results workbook.

One note on the code: the architecture cells here are read straight out of the training script, so they
cannot drift from what was actually trained. The only edit to your code is forced by the framework
version, where Keras 3 removed the `alpha` argument of LeakyReLU, so it reads `negative_slope=0.1`.
That is the same value under the new name.

## Setup

In [ ]:
# Setup: run this first. It loads the libraries and fixes the seed.
import os
import random
import time

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
import keras_hub
from sklearn.metrics import (accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_fscore_support)

SEED = 42
NUM_CLASSES = 22        # 10 defect classes from the OOI set plus 12 from Lens Presentation

# Only needed for the optional training cell at the end.
DATA_DIR = "/path/to/dataset"   # folder containing train/ and test/


def apply_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


apply_seed()
print("TensorFlow", tf.__version__, "| classes:", NUM_CLASSES)

## 1. ResNet50 MultiLevel MultiScale  (your Architecture 1)

An Inception block on each of the L3, L4 and L5 taps. The three levels are brought to a common size, concatenated, pooled, and classified through the same head in every cell of this notebook.

Accuracy on the 22-class set: **88.05**. Parameters: 58,547,094.

In [ ]:
def inception_block(inputs, filters, name=None):
    """GoogLeNet-style Inception block (FMD arch 1)."""
    branch1 = tf.keras.layers.Conv2D(
        filters=filters["branch1"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch1_1x1")(inputs)

    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_reduce"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch2_reduce")(inputs)
    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_out"], kernel_size=(3, 3), padding="same",
        activation="relu", name=f"{name}_branch2_3x3")(branch2)

    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_reduce"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch3_reduce")(inputs)
    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_out"], kernel_size=(5, 5), padding="same",
        activation="relu", name=f"{name}_branch3_5x5")(branch3)

    branch4 = tf.keras.layers.MaxPooling2D(
        pool_size=(3, 3), strides=(1, 1), padding="same",
        name=f"{name}_branch4_pool")(inputs)
    branch4 = tf.keras.layers.Conv2D(
        filters=filters["branch4_out"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch4_1x1")(branch4)

    return tf.keras.layers.Concatenate(axis=-1, name=f"{name}_concat")(
        [branch1, branch2, branch3, branch4])


def arch_resnet50_inception(num_classes):
    """ResNet50 MultiLevel MultiScale (FMD arch 1)."""
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = tf.keras.applications.resnet50.preprocess_input(inputs)
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet", include_top=False, input_tensor=x)
    base_model.trainable = True

    l3 = base_model.get_layer("conv3_block4_add").output   # 28x28x512
    l4 = base_model.get_layer("conv4_block6_add").output   # 14x14x1024
    l5 = base_model.get_layer("conv5_block3_add").output   # 7x7x2048

    l3_filters = {"branch1": 64, "branch2_reduce": 96, "branch2_out": 104,
                  "branch3_reduce": 32, "branch3_out": 172, "branch4_out": 172}
    l4_filters = {"branch1": 128, "branch2_reduce": 192, "branch2_out": 208,
                  "branch3_reduce": 64, "branch3_out": 344, "branch4_out": 344}
    l5_filters = {"branch1": 256, "branch2_reduce": 384, "branch2_out": 416,
                  "branch3_reduce": 128, "branch3_out": 688, "branch4_out": 688}

    i3 = inception_block(l3, filters=l3_filters, name="inception_l3")
    i4 = inception_block(l4, filters=l4_filters, name="inception_l4")
    i5 = inception_block(l5, filters=l5_filters, name="inception_l5")

    x3 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(i3)
    x3 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x3)

    x4 = i4

    x5 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(i5)
    x5 = tf.keras.layers.Conv2DTranspose(
        filters=2048, kernel_size=(2, 2), strides=(2, 2), padding="same")(x5)
    x5 = tf.keras.layers.BatchNormalization()(x5)
    x5 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(x5)

    fusion = tf.keras.layers.Concatenate(axis=-1)([x3, x4, x5])

    x = tf.keras.layers.GlobalAveragePooling2D()(fusion)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(2048, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(1024, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)


model = arch_resnet50_inception(NUM_CLASSES)
model.summary()

## 2. ResNet50 MultiLevel Attention  (your Architecture 2)

The same three taps, each passed through a Squeeze-and-Excitation block first, so the channels are re-weighted before fusion.

Accuracy on the 22-class set: **87.37**. Parameters: 50,544,246.

In [ ]:
def se_block(inputs, reduction=16, name=None):
    """Squeeze-and-Excitation block (FMD arch 2)."""
    channels = int(inputs.shape[-1])
    x = tf.keras.layers.GlobalAveragePooling2D(
        name=f"{name}_gap" if name else None)(inputs)
    x = tf.keras.layers.Dense(
        channels // reduction, activation="relu", use_bias=True,
        name=f"{name}_fc1" if name else None)(x)
    x = tf.keras.layers.Dense(
        channels, activation="sigmoid", use_bias=True,
        name=f"{name}_fc2" if name else None)(x)
    x = tf.keras.layers.Reshape(
        (1, 1, channels), name=f"{name}_reshape" if name else None)(x)
    return tf.keras.layers.Multiply(
        name=f"{name}_scale" if name else None)([inputs, x])


def arch_resnet50_se(num_classes):
    """ResNet50 MultiLevel Attention (FMD arch 2)."""
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = tf.keras.applications.resnet50.preprocess_input(inputs)
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet", include_top=False, input_tensor=x)
    base_model.trainable = True

    l3 = base_model.get_layer("conv3_block4_add").output   # 28x28x512
    l4 = base_model.get_layer("conv4_block6_add").output   # 14x14x1024
    l5 = base_model.get_layer("conv5_block3_add").output   # 7x7x2048

    l3_se = se_block(l3, reduction=16, name="se3")
    l4_se = se_block(l4, reduction=16, name="se4")
    l5_se = se_block(l5, reduction=16, name="se5")

    x3 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(l3_se)
    x3 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x3)

    x4 = l4_se

    x5 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(l5_se)
    x5 = tf.keras.layers.Conv2DTranspose(
        filters=2048, kernel_size=(2, 2), strides=(2, 2), padding="same")(x5)
    x5 = tf.keras.layers.BatchNormalization()(x5)
    x5 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(x5)

    fusion = tf.keras.layers.Concatenate(axis=-1)([x3, x4, x5])

    x = tf.keras.layers.GlobalAveragePooling2D()(fusion)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(2048, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(1024, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)


model = arch_resnet50_se(NUM_CLASSES)
model.summary()

## 3. Inception with attention inside every branch  (your Architecture 3)

Your Architecture 1 with a Squeeze-and-Excitation block inside each Inception branch, so attention acts before the concatenation rather than after it.

Accuracy on the 22-class set: **86.01**. Parameters: 58,744,051.

In [ ]:
def se_block(inputs, reduction=16, name=None):
    """Squeeze-and-Excitation block (FMD arch 2)."""
    channels = int(inputs.shape[-1])
    x = tf.keras.layers.GlobalAveragePooling2D(
        name=f"{name}_gap" if name else None)(inputs)
    x = tf.keras.layers.Dense(
        channels // reduction, activation="relu", use_bias=True,
        name=f"{name}_fc1" if name else None)(x)
    x = tf.keras.layers.Dense(
        channels, activation="sigmoid", use_bias=True,
        name=f"{name}_fc2" if name else None)(x)
    x = tf.keras.layers.Reshape(
        (1, 1, channels), name=f"{name}_reshape" if name else None)(x)
    return tf.keras.layers.Multiply(
        name=f"{name}_scale" if name else None)([inputs, x])


def inception_attention_block(inputs, filters, name=None):
    """GoogLeNet-style Inception block with Squeeze-and-Excitation attention on
    every branch (FMD arch 3). The four branches are the same as
    inception_block; each branch output is passed through se_block(reduction=16).
    Spatial dimensions are preserved.
    """
    # Branch 1: 1x1
    branch1 = tf.keras.layers.Conv2D(
        filters=filters["branch1"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch1_1x1")(inputs)
    branch1 = se_block(branch1, reduction=16, name=f"{name}_branch1_se")

    # Branch 2: 1x1 -> 3x3
    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_reduce"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch2_reduce")(inputs)
    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_out"], kernel_size=(3, 3), padding="same",
        activation="relu", name=f"{name}_branch2_3x3")(branch2)
    branch2 = se_block(branch2, reduction=16, name=f"{name}_branch2_se")

    # Branch 3: 1x1 -> 5x5
    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_reduce"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch3_reduce")(inputs)
    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_out"], kernel_size=(5, 5), padding="same",
        activation="relu", name=f"{name}_branch3_5x5")(branch3)
    branch3 = se_block(branch3, reduction=16, name=f"{name}_branch3_se")

    # Branch 4: 3x3 MaxPool -> 1x1
    branch4 = tf.keras.layers.MaxPooling2D(
        pool_size=(3, 3), strides=(1, 1), padding="same",
        name=f"{name}_branch4_pool")(inputs)
    branch4 = tf.keras.layers.Conv2D(
        filters=filters["branch4_out"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch4_1x1")(branch4)
    branch4 = se_block(branch4, reduction=16, name=f"{name}_branch4_se")

    return tf.keras.layers.Concatenate(
        axis=-1, name=f"{name}_concat")([branch1, branch2, branch3, branch4])


def arch_resnet50_inception_attention(num_classes):
    """ResNet50 MultiLevel Inception with branch attention (FMD arch 3:
    "ResNet50 Architecture MultiLevel Attention" in FMD_ResNet50Architectures).
    Same Inception fusion as arch 1, with SE attention inside every branch."""
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = tf.keras.applications.resnet50.preprocess_input(inputs)
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet", include_top=False, input_tensor=x)
    base_model.trainable = True

    l3 = base_model.get_layer("conv3_block4_add").output   # 28x28x512
    l4 = base_model.get_layer("conv4_block6_add").output   # 14x14x1024
    l5 = base_model.get_layer("conv5_block3_add").output   # 7x7x2048

    l3_filters = {
        "branch1": 64,
        "branch2_reduce": 96,
        "branch2_out": 104,
        "branch3_reduce": 32,
        "branch3_out": 172,
        "branch4_out": 172,
    }
    l4_filters = {
        "branch1": 128,
        "branch2_reduce": 192,
        "branch2_out": 208,
        "branch3_reduce": 64,
        "branch3_out": 344,
        "branch4_out": 344,
    }
    l5_filters = {
        "branch1": 256,
        "branch2_reduce": 384,
        "branch2_out": 416,
        "branch3_reduce": 128,
        "branch3_out": 688,
        "branch4_out": 688,
    }

    i3 = inception_attention_block(
        l3, filters=l3_filters, name="inception_att_l3")     # 28x28x512
    i4 = inception_attention_block(
        l4, filters=l4_filters, name="inception_att_l4")     # 14x14x1024
    i5 = inception_attention_block(
        l5, filters=l5_filters, name="inception_att_l5")     # 7x7x2048

    x3 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(i3)
    x3 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x3)

    x4 = i4

    x5 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(i5)
    x5 = tf.keras.layers.Conv2DTranspose(
        filters=2048, kernel_size=(2, 2), strides=(2, 2), padding="same")(x5)
    x5 = tf.keras.layers.BatchNormalization()(x5)
    x5 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(x5)

    fusion = tf.keras.layers.Concatenate(axis=-1)([x3, x4, x5])

    x = tf.keras.layers.GlobalAveragePooling2D()(fusion)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(2048, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(1024, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)


model = arch_resnet50_inception_attention(NUM_CLASSES)
model.summary()

## 4. Inception with a refined decode path  (your Architecture 4)

Your Architecture 1 with a different decoder for the deepest level: bicubic upsampling, then a depthwise spatial refinement and a 1x1 channel mixer, instead of a transposed convolution.

Accuracy on the 22-class set: **88.05**. Parameters: 45,988,758.

In [ ]:
def inception_block(inputs, filters, name=None):
    """GoogLeNet-style Inception block (FMD arch 1)."""
    branch1 = tf.keras.layers.Conv2D(
        filters=filters["branch1"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch1_1x1")(inputs)

    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_reduce"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch2_reduce")(inputs)
    branch2 = tf.keras.layers.Conv2D(
        filters=filters["branch2_out"], kernel_size=(3, 3), padding="same",
        activation="relu", name=f"{name}_branch2_3x3")(branch2)

    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_reduce"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch3_reduce")(inputs)
    branch3 = tf.keras.layers.Conv2D(
        filters=filters["branch3_out"], kernel_size=(5, 5), padding="same",
        activation="relu", name=f"{name}_branch3_5x5")(branch3)

    branch4 = tf.keras.layers.MaxPooling2D(
        pool_size=(3, 3), strides=(1, 1), padding="same",
        name=f"{name}_branch4_pool")(inputs)
    branch4 = tf.keras.layers.Conv2D(
        filters=filters["branch4_out"], kernel_size=(1, 1), padding="same",
        activation="relu", name=f"{name}_branch4_1x1")(branch4)

    return tf.keras.layers.Concatenate(axis=-1, name=f"{name}_concat")(
        [branch1, branch2, branch3, branch4])


def arch_resnet50_inception_refine(num_classes):
    """ResNet50 MultiLevel Inception with a refined decoded L5 path (FMD arch 4:
    the last cell of FMD_ResNet50Architectures). Same Inception fusion as arch 1,
    but the 7x7x2048 L5 feature is decoded with bicubic upsampling plus a
    depthwise spatial refinement and a 1x1 channel mixer instead of a
    transposed convolution."""
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = tf.keras.applications.resnet50.preprocess_input(inputs)
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet", include_top=False, input_tensor=x)
    base_model.trainable = True

    l3 = base_model.get_layer("conv3_block4_add").output   # 28x28x512
    l4 = base_model.get_layer("conv4_block6_add").output   # 14x14x1024
    l5 = base_model.get_layer("conv5_block3_add").output   # 7x7x2048

    l3_filters = {
        "branch1": 64,
        "branch2_reduce": 96,
        "branch2_out": 104,
        "branch3_reduce": 32,
        "branch3_out": 172,
        "branch4_out": 172,
    }
    l4_filters = {
        "branch1": 128,
        "branch2_reduce": 192,
        "branch2_out": 208,
        "branch3_reduce": 64,
        "branch3_out": 344,
        "branch4_out": 344,
    }
    l5_filters = {
        "branch1": 256,
        "branch2_reduce": 384,
        "branch2_out": 416,
        "branch3_reduce": 128,
        "branch3_out": 688,
        "branch4_out": 688,
    }

    i3 = inception_block(l3, filters=l3_filters, name="inception_l3")
    i4 = inception_block(l4, filters=l4_filters, name="inception_l4")
    i5 = inception_block(l5, filters=l5_filters, name="inception_l5")

    x3 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(i3)
    x3 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))(x3)

    x4 = i4

    # 7x7x2048 -> 14x14x2048 by bicubic upsample, then spatial refinement
    x5 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(i5)

    x5 = tf.keras.layers.UpSampling2D(
        size=(2, 2), interpolation="bicubic")(x5)

    x5 = tf.keras.layers.DepthwiseConv2D(
        kernel_size=(3, 3), padding="same", use_bias=False)(x5)
    x5 = tf.keras.layers.BatchNormalization()(x5)
    x5 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(x5)

    x5 = tf.keras.layers.Conv2D(
        filters=2048, kernel_size=(1, 1), padding="same", use_bias=False)(x5)
    x5 = tf.keras.layers.BatchNormalization()(x5)
    x5 = tf.keras.layers.LeakyReLU(negative_slope=0.1)(x5)

    fusion = tf.keras.layers.Concatenate(axis=-1)([x3, x4, x5])

    x = tf.keras.layers.GlobalAveragePooling2D()(fusion)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(2048, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(1024, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)


model = arch_resnet50_inception_refine(NUM_CLASSES)
model.summary()

## 5. ResNet18  (baseline we added)

A small backbone with a single pooling and dense head, included so the comparison covers a lighter model as well.

Accuracy on the 22-class set: **82.94**.

In [ ]:
def arch_resnet18(num_classes):
    """ResNet18 - samples/ResNet18.ipynb (keras_hub ResNetBackbone imagenet)."""
    inputs = keras.Input(shape=(224, 224, 3))
    base_model = keras_hub.models.ResNetBackbone.from_preset(
        "resnet_18_imagenet", load_weights=True)
    base_model.trainable = True
    x = base_model(inputs)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.BatchNormalization()(x)
    x = keras.layers.Dense(1024, activation=None)(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)


model = arch_resnet18(NUM_CLASSES)
model.summary()

## 6. ResNet50 plain  (baseline we added)

The plain ResNet50 classifier: pooling on the final feature map, one dense layer, softmax. No multi-level fusion, included as the reference point for the four architectures above.

Accuracy on the 22-class set: **88.40**. Parameters: 25,716,630.

In [ ]:
def arch_resnet50(num_classes):
    """ResNet50 - samples/JnJ_CNN.ipynb (Architecture 1)."""
    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = tf.keras.applications.resnet50.preprocess_input(inputs)
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet", include_top=False, input_tensor=x)
    base_model.trainable = True
    x = base_model.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dense(1024, activation=None)(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs, outputs)


model = arch_resnet50(NUM_CLASSES)
model.summary()

## Optional: run the training settings used for the results above

Set `DATA_DIR` in the setup cell, run one architecture cell, then run this one.

In [ ]:
# Optional: train and evaluate whichever architecture is in `model`, with the settings used for every
# number in this project. Set DATA_DIR above first.
EPOCHS, BATCH, WARMUP = 50, 8, True

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/train", image_size=(224, 224), batch_size=BATCH, shuffle=True, seed=SEED)
test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/test", image_size=(224, 224), batch_size=BATCH, shuffle=False)
class_names = train_ds.class_names
num_classes = len(class_names)

model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=1e-4, momentum=0.9, nesterov=True),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"])

t0 = time.perf_counter()
history = model.fit(train_ds, epochs=EPOCHS)
print(f"Training time : {time.perf_counter() - t0:.2f} seconds")

y_true = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)
n_test = sum(images.shape[0] for images, _ in test_ds)

if WARMUP:
    # Two dummy passes warm the graph; the third warms the real file pipeline and is discarded.
    # Without them a one-time start-up cost lands inside the timed prediction and inflates the
    # per-image time. Accuracy is unaffected either way.
    dummy = tf.data.Dataset.from_tensor_slices(
        (tf.random.normal((64, 224, 224, 3)), tf.zeros((64,), dtype=tf.int32))).batch(BATCH)
    model.predict(dummy, verbose=0)
    model.predict(dummy, verbose=0)
    model.predict(test_ds, verbose=0)

start = time.perf_counter()
y_pred = np.argmax(model.predict(test_ds, verbose=1), axis=1)
elapsed = time.perf_counter() - start

print(f"Overall accuracy : {accuracy_score(y_true, y_pred) * 100:.2f}%")
print(f"Inference per image : {elapsed / n_test * 1000:.3f} ms")

p, r, f1, support = precision_recall_fscore_support(
    y_true, y_pred, labels=list(range(num_classes)), zero_division=0)
for cls, pi, ri, fi, si in zip(class_names, p, r, f1, support):
    print(f"  {cls}: precision={pi*100:.2f}%  recall={ri*100:.2f}%  f1={fi*100:.2f}%  n={int(si)}")

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    cmap="Blues", xticks_rotation=90, ax=ax, colorbar=False)
plt.title("Confusion matrix")
plt.tight_layout()
plt.show()

## Open items, for when this work is picked up again

- **Save the trained model.** The training script records the metrics and a confusion matrix but not the
  weights, so a finished run cannot be reused or re-checked without retraining it.
- **Measure the cold-start inference time.** The workbook shows an estimate for it, marked with a tilde,
  rather than a measured number.
- **A second split.** Between the two splits we have compared, one class moved by more than 20 points, so a
  second split would show how much of any difference is the split rather than the model. Until then, only
  comparisons made within one split should be read as architecture results.